# TP2 · 04 · Submissão tardia (Internet **Off**, GPU)

A competição é **de código**: ao submeter, o Kaggle roda este notebook de novo, sobre o teste
**oculto** (os 650 exames públicos + ~1.517 privados). Por isso a inferência parte dos DICOMs de
`test.csv`: volume na janela CTPA → DINOv2 ViT-S/14 com LLRD (3 sementes) → calibração → 10 rótulos
→ regras de consistência → `/kaggle/working/submission.csv`.

**Inputs** (Add Input):
1. a competição **RSNA STR Pulmonary Embolism Detection**;
2. o Dataset **tp2-pesos-dinov2-llrd** (o zip `kaggle/tp2-pesos-dinov2-llrd.zip`), **público**: a
   competição só aceita dados externos livres e públicos;
3. a saída do notebook **tp2-sub-pacote** (aba *Notebooks* do Add Input).

**Settings:** Accelerator **GPU T4 x2**, Internet **Off**.

Na execução normal (Save & Run All), o relatório mostra a concordância (`parity_vs_extraction`)
com as predições feitas no treino para os 650 exames públicos: deve ficar perto de 1.


In [ ]:
# ===== TP2 · submissão tardia =====
import glob, os, subprocess, sys, time
T0 = time.time()

def achar(nome, o_que):
    # sem busca recursiva (a pasta da competição tem ~1,8 milhão de DICOMs)
    for prof in range(0, 4):
        h = sorted(glob.glob("/kaggle/input/" + "*/" * prof + nome))
        if h:
            return h[0]
    raise SystemExit(f"não encontrado: {o_que} ({nome})")

CODE = os.path.dirname(os.path.dirname(achar("code/configs/config.yaml", "código (saída do tp2-sub-pacote)")))
WH = achar("wheels", "wheels (saída do tp2-sub-pacote)")
PKGS = ["pydicom>=3.0", "pylibjpeg>=2.0", "pylibjpeg-libjpeg>=2.1", "pylibjpeg-openjpeg>=2.3", "python-gdcm>=3.0.24"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", WH, *PKGS], check=True)
PESOS = os.path.dirname(os.path.dirname(achar("ft_dinov2_s_llrd_main_s0/model.pt", "pesos (dataset tp2-pesos-dinov2-llrd)")))
RUNS = f"{PESOS}/ft_dinov2_s_llrd_main_s*"
print("código:", CODE, open(f"{CODE}/COMMIT.txt").read().strip(), "| pesos:", PESOS)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -u {CODE}/scripts/13_kaggle_infer.py --runs "{RUNS}" --out /kaggle/working --workers 4
print(f"TEMPO TOTAL: {(time.time() - T0) / 60:.1f} min")
